In [79]:
from datasets import load_dataset

funsd = load_dataset("nielsr/funsd")

FUNSD_LABELS = ["O", "question", "answer", "header", "other"]

def convert_funsd_example(example):
    boxes = []
    words = example["words"]
    bboxes = example["bboxes"]
    ner_tags = example["ner_tags"]
    
    for word, box, tag in zip(words, bboxes, ner_tags):
        boxes.append({
            "box": box,
            "label": FUNSD_LABELS[tag] if tag < len(FUNSD_LABELS) else "other",
            "text": word
        })
    return boxes

example = funsd['test'][0]
gt_boxes = convert_funsd_example(example)
gt_boxes[:5]

[{'box': [135, 345, 171, 359], 'label': 'header', 'text': 'TO:'},
 {'box': [135, 406, 194, 423], 'label': 'header', 'text': 'DATE:'},
 {'box': [611, 440, 629, 455], 'label': 'other', 'text': '3'},
 {'box': [330, 84, 363, 98], 'label': 'header', 'text': 'Fax:'},
 {'box': [136, 691, 184, 708], 'label': 'header', 'text': 'NOTE:'}]

In [80]:
import json
from datasets import load_dataset
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

funsd = load_dataset("nielsr/funsd")
test_set = funsd['test']
train_set = funsd['train']
OUTPUT_DIR = Path("C:/Users/Legion/Desktop/DocuAgent-AP/outputs")
EVAL_DIR = Path("evaluation")
EVAL_DIR.mkdir(exist_ok=True)

metrics = []

def get_gt_text(example):
    words = [w for w in example['words'] if w.strip() != ""]
    return " ".join(words)

def get_pred_text(result_json):
    pred_texts = []
    for box in result_json.get("page_1", {}).get("boxes", []):
        content = box.get("block_content")
        if content:
            if isinstance(content, dict) and "base_64_encoded" in content:
                continue
            pred_texts.append(str(content))
    return " ".join(pred_texts)

def tokenize(text):
    return text.split()

def compute_metrics(gt_text, pred_text):
    gt_tokens = tokenize(gt_text)
    pred_tokens = tokenize(pred_text)
    
    all_tokens = list(set(gt_tokens + pred_tokens))
    gt_vec = [1 if t in gt_tokens else 0 for t in all_tokens]
    pred_vec = [1 if t in pred_tokens else 0 for t in all_tokens]
    
    precision = precision_score(gt_vec, pred_vec)
    recall = recall_score(gt_vec, pred_vec)
    f1 = f1_score(gt_vec, pred_vec)
    
    return precision, recall, f1

In [81]:
metrics = []

for img_path in sorted(EVAL_DIR.glob("*.png")):
    img_name = img_path.stem
    is_train = "_train" in img_name
    index = int(img_name.replace("_train", ""))

    example = train_set[index] if is_train else test_set[index]
    gt_text = get_gt_text(example)

    if is_train:
        result_json_path = (OUTPUT_DIR / f"{index}_train" / f"{index}_train_all_results.json").resolve()
    else:
        result_json_path = (OUTPUT_DIR / str(index) / f"{index}_all_results.json").resolve()

    if not result_json_path.exists():
        print(f"[⚠️ Missing JSON] {result_json_path}")
        continue

    with open(result_json_path, "r", encoding="utf-8") as f:
        pipeline_pred = json.load(f)

    pred_text = get_pred_text(pipeline_pred)
    if not pred_text.strip():
        continue

    precision, recall, f1 = compute_metrics(gt_text, pred_text)
    metrics.append((img_name, precision, recall, f1))
    print(f"{img_name}: P={precision:.2f}, R={recall:.2f}, F1={f1:.2f}")

arr = np.array([m[1:] for m in metrics])
print(f"\nAverage Precision={arr[:,0].mean():.2f}, Recall={arr[:,1].mean():.2f}, F1={arr[:,2].mean():.2f}")


0: P=0.54, R=0.47, F1=0.50
0_train: P=0.77, R=0.72, F1=0.74
1: P=0.61, R=0.42, F1=0.49
100_train: P=0.76, R=0.74, F1=0.75
101_train: P=0.98, R=0.91, F1=0.95
102_train: P=0.76, R=0.75, F1=0.76
114_train: P=0.69, R=0.71, F1=0.70
115_train: P=0.79, R=0.77, F1=0.78
125_train: P=0.84, R=0.62, F1=0.72
126_train: P=0.84, R=0.89, F1=0.86
129_train: P=0.82, R=0.66, F1=0.73
12_train: P=0.82, R=0.91, F1=0.86
130_train: P=0.79, R=0.82, F1=0.80
133_train: P=0.73, R=0.73, F1=0.73
135_train: P=0.78, R=0.72, F1=0.75
137_train: P=0.75, R=0.69, F1=0.71
138_train: P=0.87, R=0.84, F1=0.85
14: P=0.55, R=0.50, F1=0.52
140_train: P=0.75, R=0.68, F1=0.71
142_train: P=0.75, R=0.66, F1=0.70
15: P=0.06, R=0.01, F1=0.01
15_train: P=0.82, R=0.72, F1=0.77
2: P=0.81, R=0.75, F1=0.78
20_train: P=0.78, R=0.74, F1=0.76
21_train: P=0.83, R=0.76, F1=0.79
22_train: P=0.84, R=0.81, F1=0.82
24: P=0.75, R=0.78, F1=0.76
24_train: P=0.80, R=0.66, F1=0.72
27: P=0.70, R=0.71, F1=0.71
28_train: P=0.77, R=0.72, F1=0.74
29: P=0.57,